# Prepare NN-duel forecasts conditioned on the observer's current skill

Run this notebook before **13_nn_duel_skill_forecasting.ipynb**. It reads the most
recently published **complete** flight series from **11_nn_vs_nn_training_data.ipynb**,
prepares both aircraft perspectives, and saves a reusable dataset. On this branch
the completed release is `shared/corpora/nn-duels-20k-v1/eee3474c5493`:
**10,000 physical flights / 20,000 perspective episodes**. An explicit source
override is available below. Tracked releases are ranked by their manifest's Git
commit time; unpublished local packages fall back to their manifest modification time.

For **A observing B**, each history frame contains A's existing **51 observations
+ three causal acceleration features**, followed by an indicator for **A's current
selected skill**. The labels remain **B's selected skill at +1, +2 and +5 seconds**.
The same network input schema supports B observing A. These recordings contain
one selected catalogue skill per aircraft at a time; the indicator vocabulary
retains every catalogue skill, including those absent from this release. It does
not invent simultaneous skills or infer physical execution from catalogue names.

**Run All downloads the required Git LFS payloads and prepares data; it does not
collect new flights or train models.** Install the repository's `.[ml]` dependencies
and a Jupyter/IPython kernel first (Python 3.10+, NumPy, pandas, PyArrow, PyTorch).
The archived raw release is about **10.3 GB**; the 42.2 million-frame, 87-feature
float32 array is about **13.7 GiB**, plus labels, indices and reports. Allow at
least 30 GB of free disk space before downloading and preparing the full release.
Preparation reads one shard at a time and writes a memory map rather than expanding
every overlapping window. Every executable cell reports CPU and wall time.


In [ ]:
%%time
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_skill_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_OVERRIDE = os.environ.get("BVR_NN_FORECAST_SOURCE")  # Full path, or repository-relative.
SOURCE = duel_data.discover_corpus(ROOT, SOURCE_OVERRIDE)
SOURCE_MANIFEST = read_json(SOURCE / "manifest.json")
SOURCE_DIGEST = sha256(SOURCE / "manifest.json")
PREPARATION_DIGEST = sha256(duel_data.__file__)
FEATURE_HELPERS_DIGEST = sha256(ROOT / "bvr_behavior_prediction/paired_skill_forecasting.py")
DATA_DEFAULT = (EXPERIMENT_ROOT / "data" /
                f"{SOURCE.name}_{SOURCE_DIGEST[:12]}_{PREPARATION_DIGEST[:8]}_{FEATURE_HELPERS_DIGEST[:8]}")
DATA = Path(os.environ.get("BVR_NN_FORECAST_DATA", str(DATA_DEFAULT))).expanduser().resolve()
DOWNLOAD_LFS = os.environ.get("BVR_NN_FORECAST_DOWNLOAD", "1") == "1"
GIT_EXECUTABLE = None  # e.g. r"C:\Program Files\Git\cmd\git.exe" on Windows.
SPLIT_SEED = 1729
print("Source:", SOURCE)
print("Complete:", SOURCE_MANIFEST["complete"], "| physical flights:",
      SOURCE_MANIFEST["physical_flights"], "| perspectives:", SOURCE_MANIFEST["perspective_episodes"])
print("Source manifest SHA-256:", SOURCE_DIGEST)
print("Prepared output:", DATA)


## Alignment and forecasting contract

* Histories contain **51 chronological frames at 10 Hz**, spanning five seconds.
  Five additional past frames support backward 0.5-second acceleration. The first
  forecast origin is 6 seconds, with a one-second stride; no history crosses a flight.
* For observer A, its own A decision comes from **B's `target_decisions` file**.
  Actual aircraft IDs and episode keys are checked before alignment. A decision
  at tick 60 applies at ticks 60–69; the decision at 70 cannot affect earlier inputs.
  Missing own decision intervals fail preparation rather than being filled indefinitely.
  This is the pilot's selected skill known when issuing guidance at the frame timestamp.
* Future opponent labels require an **exact recorded decision**. Missing +1/+2/+5
  labels are masked independently with `-1`; there is no extrapolation beyond termination.
  Opponent decisions are labels only. Pilot/checkpoint IDs and outcomes are metadata.
* **40% train / 30% test / 30% validation**, grouped by `scenario_group`, keep
  both perspectives of every physical flight together. Seeded range strata preserve
  starting-range balance while balancing per-skill/horizon episode and window support.
  Continuous NN-duel ranges require strata in place of 09b's discrete range quotas.
  Rare skills and indivisible groups can prevent exact class proportions; inspect the audit.
* Scaling is fitted on unique frames used by **training histories only**. The
  appended skill indicators stay binary (mean 0, scale 1). No future labels or
  validation frames enter normalization. Earlier corpus files and 09b caches are preserved.


In [ ]:
%%time
if DOWNLOAD_LFS:
    duel_data.download_corpus(ROOT, SOURCE, git_executable=GIT_EXECUTABLE)

analysis = duel_data.analyze_corpus(ROOT, SOURCE)
support = duel_data.skill_support_table(analysis)
print(f"Surveyed {len(analysis['episodes']):,} perspectives before assigning splits.")
display(analysis["episodes"].groupby(["observer_id", "family"]).agg(
    perspectives=("episode_id", "size"), mean_duration_s=("duration_s", "mean")))
with pd.option_context("display.max_rows", None):
    display(support)
print("Zero-window catalogue labels:", sorted(set(analysis["classes"]) -
      set(support.loc[support.windows.gt(0), "skill"])))


In [ ]:
%%time
if (DATA / "manifest.json").exists():
    manifest = read_json(DATA / "manifest.json")
    if SOURCE_DIGEST not in manifest["source_manifests"].values():
        raise ValueError("The existing dataset belongs to another source; choose a new DATA path.")
    if manifest["source_code_sha256"] != PREPARATION_DIGEST:
        raise ValueError("Preparation code changed; choose a fresh DATA path.")
    if manifest["feature_helpers_sha256"] != FEATURE_HELPERS_DIGEST:
        raise ValueError("Feature helper code changed; choose a fresh DATA path.")
    if manifest["split_seed"] != SPLIT_SEED:
        raise ValueError("Split seed changed; choose a fresh DATA path.")
    print("Reusing the verified prepared dataset:", DATA)
else:
    manifest = duel_data.prepare_dataset(ROOT, DATA, source=SOURCE,
                                         split_seed=SPLIT_SEED, analysis=analysis)
data = duel_data.NNDuelData(DATA)  # Verify stored file hashes and the input/target contract.
assert data.features.shape[1] == 54 + len(data.classes)
assert data.manifest["features"] == manifest["features"]
display(pd.DataFrame(manifest["counts"]).T)
display(pd.Series({"frames": len(data.features), "features_per_frame": data.features.shape[1],
                   "feature_array_GiB": data.features.nbytes / 2**30,
                   "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS}))


## Inspect the split and the appended inputs

The support tables retain zero-support classes. A missing class in validation is
visible here and is not repaired by changing the held-out partition after training.
The example below checks a normalized history from a training sample: its appended
indicators still identify the observing aircraft's own skill at every frame.


In [ ]:
%%time
assignments = pd.read_csv(DATA / "split_assignments.csv")
assert assignments.groupby("scenario_group").split.nunique().eq(1).all()
assert assignments.groupby("episode_id").observer_id.nunique().eq(2).all()
assert assignments.groupby("episode_id").split.nunique().eq(1).all()
display(assignments.groupby(["split", "observer_id"]).size().unstack(fill_value=0))
split_support = pd.read_csv(DATA / "skill_support_by_split.csv")
with pd.option_context("display.max_rows", None):
    display(split_support.pivot(index=["horizon_s", "skill"], columns="split",
                                values=["episodes", "windows", "scenario_groups"]))
sample_id = int(data.indices["train"][0])
x, y = data.batch(np.asarray([sample_id]), "cpu")
own = x[0, :, 54:].numpy()
assert np.isin(own, [0, 1]).all() and np.all(own.sum(axis=1) == 1)
assert np.all(data.mean[54:] == 0) and np.all(data.std[54:] == 1)
sample = data.samples.iloc[sample_id]
display(sample)
display(pd.DataFrame({
    "time_s": np.arange(sample.step - HISTORY_STEPS + 1, sample.step + 1) * .1,
    "observer_current_skill": np.asarray(data.classes)[own.argmax(axis=1)]}))
display(pd.Series({f"opponent_at_+{h}s": data.classes[int(t)] if t >= 0 else "MASKED"
                   for h, t in zip(HORIZONS, y[0].numpy())}))

# Written only after preparation and verification finish. Notebook 13 ingests this exact dataset.
write_json(EXPERIMENT_ROOT / "latest_dataset.json", {
    "directory": str(DATA), "manifest_sha256": sha256(DATA / "manifest.json"),
    "source": str(SOURCE), "source_manifest_sha256": SOURCE_DIGEST})
print("Ready for notebook 13:", DATA)
